In [ ]:
import pydicom
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

main_folder = Path(r"\\pom\pmcommon\_PM.Private\CSIC\Moulin\4D Flow Data")
TARGET_SERIES_DESCRIPTION = "4D"
MATCH_MODE = "contains"

def process_patient(patient_dir):
    matches = []
    found_any = False
    for sequence_dir in sorted(patient_dir.iterdir()):
        if not sequence_dir.is_dir():
            continue
        dcm_files = list(sequence_dir.glob("*.dcm"))
        if not dcm_files:
            continue
        try:
            ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
            series_desc = getattr(ds, "SeriesDescription", "") or ""
            match = (
                TARGET_SERIES_DESCRIPTION.lower() in series_desc.lower()
                if MATCH_MODE == "contains"
                else series_desc.lower() == TARGET_SERIES_DESCRIPTION.lower()
            )
            if match:
                matches.append({
                    "Patient Name": getattr(ds, "PatientName", None),
                    "patient_folder": patient_dir.name,
                    "sequence_folder": sequence_dir.name,
                    "scan_path": str(sequence_dir),
                    "StudyDate": getattr(ds, "StudyDate", None),   # <-- added, e.g. "20240115"
                    "num_files": len(dcm_files),
                    "PatientMRN": getattr(ds, "PatientID", None),
                    "SeriesDescription": series_desc,
                    "Modality": getattr(ds, "Modality", None),
                })
                found_any = True
        except Exception as e:
            print(f"Skipped {sequence_dir}: {e}")
    return matches, (patient_dir.name if not found_any else None)

patient_dirs = [d for d in main_folder.iterdir() if d.is_dir()]

records = []
no_match = []

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(process_patient, pd_dir): pd_dir for pd_dir in patient_dirs}
    for future in tqdm(as_completed(futures), total=len(futures)):
        matches, unmatched_patient = future.result()
        records.extend(matches)
        if unmatched_patient:
            no_match.append(unmatched_patient)

df = pd.DataFrame(records)
print(f"Found {len(df)} matching sequences")

df = df.rename(columns={
    'patient_folder': 'Accession Number',
    'sequence_folder': 'Sequence Folder',
    'num_files': 'Image Count',
    'SeriesDescription' : 'Series Description',
})

(df.style
 .set_properties(**{
     'text-align': 'center',
     'font-family': 'Arial, sans-serif',
     'border': '1px solid white'
 }))

# Group rows by patient folder name
df["Patient Name"] = df["Patient Name"].astype(str)
df = df.sort_values("Patient Name").reset_index(drop=True)
df.to_csv("4dflow_scan_metadata.csv", index=False)
print(f"Saved {len(df)} rows to 4dflow_scan_metadata.csv")

  0%|          | 0/141 [00:06<?, ?it/s]


In [7]:
import pandas as pd
df_sorted = pd.read_csv("4dflow_scan_metadata.csv")
print(f"Loaded {len(df_sorted)} rows")

Loaded 410 rows


In [8]:
%matplotlib inline

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np

# Prep display data by creating a clean copy of df_sorted, reformat patient name "Last, First"
df_display = df_sorted.reset_index(drop=True)
df_display["Patient Name"] = df_display["Patient Name"].str.replace("^", ", ", regex=False)

patient_cols = ["Patient Name", "PatientMRN"]
hidden_cols = ["scan_path", "StudyDate"]
scan_cols = [c for c in df_display.columns if c not in patient_cols + hidden_cols]

cell_layout = widgets.Layout(width="150px", overflow="hidden")
checkbox_layout = widgets.Layout(width="60px")
row_layout = widgets.Layout(width="100%", min_height="35px", height="35px", padding="4px 0px", overflow="hidden")

def centered(text, bold=False):
    weight = "font-weight:bold;" if bold else ""
    return widgets.HTML(value=f"<div style='text-align:center; {weight}' title='{text}'>{text}</div>", layout=cell_layout)

search_box = widgets.Text(
    placeholder="Search by patient name or MRN...",
    layout=widgets.Layout(width="300px", margin="20px 0px 20px 10px")
)
search_box.add_class("mytext")

display(HTML("""
<style>
.mytext > .widget-label { font-size: 20px; }
.mytext > input[type="text"] {
    font-family: 'Calibri', sans-serif;
    font-size: 16px;
    border: 2px solid #36454F !important;
    border-radius: 4px;
}
</style>
"""))

def build_header():
    return widgets.HBox(
        [widgets.HTML(value="<div style='text-align:center; font-weight:bold;'>Select</div>", layout=checkbox_layout)] +
        [centered(col, bold=True) for col in scan_cols],
        layout=widgets.Layout(width="100%", border="0px 0px 2px 0px solid black", padding="4px 0px")
    )

selected_list_container = widgets.VBox(
    [],
    layout=widgets.Layout(display="flex", flex_flow="column", width="100%")
)

def update_selected_summary():
    selected_indices = [i for i, cb in checkboxes.items() if cb.value]
    if not selected_indices:
        selected_list_container.children = [widgets.HTML(value="<i>No scans selected.</i>")]
        return

    rows = []
    for i in selected_indices:
        row = df_display.loc[i]
        seq_label = row.get("Sequence Folder", row.get(scan_cols[0], ""))
        label_text = f"{row['Patient Name']} | {seq_label}"
        label = widgets.HTML(
            value=f"<span style='font-size:13px;'>{label_text}</span>",
            layout=widgets.Layout(width="300px", overflow="hidden")
        )
        remove_btn = widgets.Button(
            description="x",
            layout=widgets.Layout(width="28px", height="28px"),
            tooltip="Remove"
        )

        def make_remove_handler(idx):
            def handler(b):
                checkboxes[idx].value = False
            return handler

        remove_btn.on_click(make_remove_handler(i))
        rows.append(widgets.HBox(
            [label, remove_btn],
            layout=widgets.Layout(align_items="center", padding="2px 0px", min_height="30px", flex="0 0 auto")
        ))

    selected_list_container.children = rows

checkboxes = {}
patient_sections = {}

def format_study_date(date_val):
    try:
        return datetime.strptime(str(date_val), "%Y%m%d").strftime("%m/%d/%Y")
    except (ValueError, TypeError):
        return str(date_val)

def scan_sort_key(sequence_folder):
    try:
        return int(str(sequence_folder).split("_")[0])
    except (ValueError, IndexError):
        return float("inf")

for patient_key, group in df_display.groupby(patient_cols, sort=False):
    name, mrn = patient_key if isinstance(patient_key, tuple) else (patient_key, "")
    title = f"{name} | MRN: {mrn} ({len(group)} scans)"
    rows = [build_header()]
    for date_val, date_group in group.sort_values("StudyDate", ascending=False).groupby("StudyDate", sort=False):
        rows.append(widgets.HTML(
            value=f"<div style='font-weight:bold; padding:8px 0 2px;'>{format_study_date(date_val)}</div>"
        ))
        date_group_sorted = date_group.assign(
            _scan_num=date_group["Sequence Folder"].apply(scan_sort_key)
        ).sort_values("_scan_num")
        for i, row in date_group_sorted.iterrows():
            cb = widgets.Checkbox(value=False, indent=False, layout=checkbox_layout)
            cb.observe(lambda change: update_selected_summary(), names="value")
            checkboxes[i] = cb
            cells = [cb] + [centered(row[col]) for col in scan_cols]
            rows.append(widgets.HBox(cells, layout=row_layout))

    patient_sections[patient_key] = (title, widgets.VBox(rows))

update_selected_summary()

accordion = widgets.Accordion(children=[body for _, body in patient_sections.values()])
for idx, (title, _) in enumerate(patient_sections.values()):
    accordion.set_title(idx, title)

accordion_container = widgets.VBox(
    [accordion],
    layout=widgets.Layout(height="400px", width="100%", overflow="auto", border="1px solid lightgray")
)

def rebuild_accordion(filtered_keys):
    children = [patient_sections[k][1] for k in filtered_keys]
    accordion.children = children
    for idx, k in enumerate(filtered_keys):
        accordion.set_title(idx, patient_sections[k][0])

def on_search_change(change):
    query = change["new"].strip().lower()
    if not query:
        rebuild_accordion(list(patient_sections.keys()))
    else:
        mask = (
            df_display["Patient Name"].astype(str).str.lower().str.contains(query, na=False) |
            df_display["PatientMRN"].astype(str).str.lower().str.contains(query, na=False)
        )
        matching_patients = df_display.loc[mask, patient_cols].drop_duplicates()
        filtered_keys = [tuple(x) for x in matching_patients.itertuples(index=False)]
        rebuild_accordion([k for k in patient_sections.keys() if k in filtered_keys])

search_box.observe(on_search_change, names="value")

selected_container = widgets.VBox(
    [selected_list_container],
    layout=widgets.Layout(height="100px", width="100%", overflow="auto", border="1px solid lightgray", padding="6px")
)

run_button = widgets.Button(description="Run", button_style="primary")
output = widgets.Output()

def first_nonzero_vel(scan):
    for v in range(scan.shape[3]):
        if scan[:, :, :, v, :].max() > 0:
            return v
    return 0

def phase_to_velocity(phase_pixels, venc, grayscale_mid=2048):
    return (phase_pixels.astype(float) - grayscale_mid) / grayscale_mid * venc

def run_function(scans):
    mag_idxs = [idx for idx, s in scans.items() if s["type"] == "M"]
    phase_idxs = [idx for idx, s in scans.items() if s["type"] == "P"]

    if not mag_idxs or not phase_idxs:
        print("Need at least one magnitude and one phase scan selected.")
        return

    mag_arr = scans[mag_idxs[0]]["array"]
    n_slices, n_time = mag_arr.shape[2], mag_arr.shape[4]
    v_mag = first_nonzero_vel(mag_arr)
    slice_slider = widgets.IntSlider(min=0, max=n_slices - 1, value=n_slices // 2, description="Slice:", continuous_update=False)
    time_slider = widgets.IntSlider(min=1, max=n_time - 1, value=0, description="Time:", continuous_update=False)
    venc_box = widgets.FloatText(value=scans[phase_idxs[0]]["venc"] or 100.0, description="Venc (cm/s):")
    out = widgets.Output()

    def render(change=None):
        phase_idx = phase_idxs[0]
        s, t = slice_slider.value, time_slider.value
        phase_arr = scans[phase_idx]["array"]
        v_phase = first_nonzero_vel(phase_arr)
        venc = venc_box.value
        mag_img = mag_arr[:, :, s, v_mag, t]
        velocity_img = phase_to_velocity(phase_arr[:, :, s, v_phase, t], venc)

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(mag_img, cmap="gray")
        im = ax.imshow(velocity_img, cmap="seismic", alpha=0.5, vmin=-venc, vmax=venc)
        ax.set_title(f"Phase idx {phase_idx} - slice {s}, time {t}, venc {venc} cm/s")
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Velocity (cm/s)")
        plt.tight_layout()

        with out:
            clear_output(wait=True)
            plt.show()

    slice_slider.observe(render, names="value")
    time_slider.observe(render, names="value")
    venc_box.observe(render, names="value")
    render()
    display(widgets.VBox([slice_slider, time_slider, venc_box, out]))

def on_run_click(b):
    with output:
        clear_output()
        selected_indices = [i for i, cb in checkboxes.items() if cb.value]
        if not selected_indices:
            print("No scans selected.")
            return
        triage(df_display.loc[selected_indices])

run_button.on_click(on_run_click)

table = widgets.VBox([
    search_box,
    accordion_container,
    widgets.HTML(value="<b>Selected scans</b>"),
    selected_container,
    run_button,
    output,
])

display(table)

In [10]:
import pydicom
import numpy as np
from pathlib import Path
from nibabel.nicom import csareader

SCAN_PATH_COL = "scan_path"

def get_dicom_files(folder):
    return sorted(Path(folder).glob("*.dcm"))

# Returns 'M', 'P', or None - standard DICOM tag
def get_image_type_tag(folder):
    files = get_dicom_files(folder)
    if not files:
        return None
    dcm = pydicom.dcmread(files[0], stop_before_pixels=True)
    image_type = getattr(dcm, "ImageType", [])
    if "P" in image_type:
        return "P"
    elif "M" in image_type:
        return "M"
    return None

# Tries the standard FlowVenc CSA tag. Returns None if not found
def get_venc(folder, files_to_check=10):
    files = get_dicom_files(folder)
    for f in files[:files_to_check]:
        dcm = pydicom.dcmread(f, stop_before_pixels=True)
        csa = csareader.get_csa_header(dcm, "image")
        if csa is None:
            continue
        venc_items = csa["tags"].get("FlowVenc", {}).get("items", [])
        if venc_items:
            return float(venc_items[0])
    return None

# Reconstructs (rows, cols, n_slices, n_vel, n_times) using Siemens ICE_Dims
def sort_dicom_series(folder):
    files = get_dicom_files(folder)

    dim_time = 3 - 1
    dim_phase = 4 - 1
    dim_slice = 7 - 1

    images, ice_vals_list, skipped = [], [], []
    for fname in files:
        dcm = pydicom.dcmread(fname)
        csa = csareader.get_csa_header(dcm, "image")
        if csa is None:
            skipped.append(fname.name)
            continue
        ice_raw = csa["tags"]["ICE_Dims"]["items"][0]
        parts = ice_raw.split("_")
        vals = []
        for p in parts:
            try:
                vals.append(int(p.strip()))
            except ValueError:
                vals.append(0)
        images.append(dcm.pixel_array.astype(float))
        ice_vals_list.append(vals)

    if skipped:
        print(f"[warn] skipped {len(skipped)} file(s) with no CSA header in {folder}")
    if not images:
        raise ValueError(f"No valid CSA headers found in {folder}")

    ice_vals = np.array(ice_vals_list)
    n_slices = int(ice_vals[:, dim_slice].max()) + 1
    n_times = int(ice_vals[:, dim_time].max()) + 1
    n_vel = int(ice_vals[:, dim_phase].max()) + 1
    rows, cols = images[0].shape

    Dcm = np.zeros((rows, cols, n_slices, n_vel, n_times), dtype=float)
    for img, vals in zip(images, ice_vals):
        s, t, v = vals[dim_slice], vals[dim_time], vals[dim_phase]
        Dcm[:, :, s, v, t] = img

    return Dcm

def first_nonzero_vel(scan):
    for v in range(scan.shape[3]):
        if scan[:, :, :, v, :].max() > 0:
            return v
    return 0

def triage(selected_df):
    """Loads every selected scan, tags it as magnitude/phase, no pairing assumptions."""
    scans = {}
    for idx, row in selected_df.iterrows():
        folder = row[SCAN_PATH_COL]
        img_type = get_image_type_tag(folder)
        if img_type is None:
            print(f"[warn] Could not determine ImageType for idx={idx} - skipping.")
            continue
        arr = sort_dicom_series(folder)
        venc = get_venc(folder) if img_type == "P" else None
        scans[idx] = {"array": arr, "type": img_type, "venc": venc, "folder": folder}

    if not scans:
        print("No usable scans loaded.")
        return

    run_function(scans)